# 05_01 - Baseline Forecasting: Interpretable Reference Models
## 1. Methodology Overview

This notebook loads the real project splits from `data/processed/` and trains the baseline forecasting suite used in the modeling layer.
The implementation lives in `src/models/baseline_models.py` and `src/models/train_model.py`.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

The aim is not to invent a new benchmark, but to reproduce the project's own baseline logic on the data already prepared by the pipeline.

In [4]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.constants import DATE_COLUMN, TARGET_COLUMN

train_path = Path('../../data/processed/train_features.csv')
validation_path = Path('../../data/processed/validation_features.csv')

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

train_df[DATE_COLUMN] = pd.to_datetime(train_df[DATE_COLUMN])
validation_df[DATE_COLUMN] = pd.to_datetime(validation_df[DATE_COLUMN])

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')
print(f'Train date range: {train_df[DATE_COLUMN].min().date()} to {train_df[DATE_COLUMN].max().date()}')
print(f'Validation date range: {validation_df[DATE_COLUMN].min().date()} to {validation_df[DATE_COLUMN].max().date()}')
print(f'Target column: {TARGET_COLUMN}')

display(train_df[[DATE_COLUMN, TARGET_COLUMN]].head())

Train features: 1461 rows x 155 columns
Validation features: 366 rows x 155 columns
Train date range: 2020-01-01 to 2023-12-31
Validation date range: 2024-01-01 to 2024-12-31
Target column: Spot_Price_SPEL


,date,Spot_Price_SPEL
0,2020-01-01,35.54
1,2020-01-02,40.00
2,2020-01-03,39.51
3,2020-01-04,35.67
4,2020-01-05,38.18


## 2. Baseline Model Suite

The baseline layer is intentionally simple and interpretable. It includes a naive last-value forecast, a seasonal naive forecast, a rolling mean, and a linear regression baseline.
The orchestration wrapper is `src/models/train_model.py`, which calls the lower-level implementations in `src/models/baseline_models.py`.

In [3]:
from src.models.train_model import train_baseline_suite

baseline_output = train_baseline_suite(train_df=train_df, eval_df=validation_df)

display(baseline_output.summary)

best_baseline = baseline_output.summary.sort_values(['rmse', 'mae']).iloc[0]
print(f"Best baseline model: {best_baseline['model_name']}")
print(f"Best baseline RMSE: {best_baseline['rmse']:.4f}")

,model_name,mae,rmse
0,naive_last_value,14.993534,21.187930
1,linear_regression_baseline,19.098751,24.073276
2,rolling_mean_7,21.366524,27.617335
3,seasonal_naive_lag_7,26.656089,34.927369


Best baseline model: naive_last_value
Best baseline RMSE: 21.1879


## 3. Handoff

This notebook provides the reference point for the rest of the modeling stage.
The next step is quantile regression, which uses the same project data but shifts the goal from point prediction to uncertainty-aware forecasting.